# 1(a). Preprocessing Functions

In [82]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sb
import numpy as np 

from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, average_precision_score, roc_curve, auc
from sklearn.metrics import precision_recall_curve, f1_score


scaling_type="StandardScaler"
# scaling_type="MinMaxScaler"
top_col_no=20
MIN_ERR=0.05

def datastat(dataframe):
    # dataframe.columns
    print("Number of attributes: ", dataframe.shape[1])
    print("Number of records: ", dataframe.shape[0])
    # print("Stats of dataset:\n",dataframe.describe())

def missing_duplicate(dataframe):
    missing=dataframe.isnull().sum()
    duplicate=dataframe.duplicated().sum()
    print("Missing:\n",missing)
    print("Duplicate: ",duplicate)

def droprows(dataframe, target_col, dup=True):
    #drop target null rows and duplicate rows
    dataframe.dropna(subset=[target_col],inplace=True) 
    # missing=dataframe.isnull().sum()
    # print("Missing:\n",missing)
    # print(dataframe.shape)

    #drop duplicate rows
    if(dup):
        dataframe.drop_duplicates(inplace=True)
    duplicate=dataframe.duplicated().sum()
    print("Duplicate: ",duplicate)

    return dataframe
    

def replace_str(dataframe):
    #replace str cols with mode of that cols
    non_numeric_cols = dataframe.select_dtypes(exclude=['number']).columns
    # print("non numeric cols: ",non_numeric_cols)

    for col in non_numeric_cols:
        mode_value = dataframe[col].mode()[0]
        # print("mode: ",mode_value)
        dataframe[col] = dataframe[col].fillna(mode_value)

    return dataframe


def replace_num(dataframe):
    #replace numeric cols with mean of that cols
    numeric_cols = dataframe.select_dtypes(include=['number']).columns
    # print("numeric cols: ",numeric_cols)

    mean_values = dataframe[numeric_cols].mean()
    # print("mean: ",mean_values)
    dataframe[numeric_cols] = dataframe[numeric_cols].fillna(mean_values)

    return dataframe

def onehot(features_non_numeric, non_numeric_cols):
    for col in non_numeric_cols: 
        features_non_numeric[col]=features_non_numeric[col].astype('category')

    print(features_non_numeric.dtypes)

    #one hot
    features_non_numeric=pd.get_dummies(features_non_numeric)
    # features_non_numeric.head(5)
    # features_non_numeric
    return features_non_numeric


#scaling func
def Scaling(features,type): 
    if type=="StandardScaler":  
        scalar=StandardScaler()
    elif type=="MinMaxScaler":
        scalar=MinMaxScaler()
    else:
        scalar=MinMaxScaler()
        
    scaled_features=scalar.fit_transform(features)
    return scaled_features

# Plot the correlation matrix
def plot_corr(correlations):
    correlation_matrix=pd.DataFrame(correlations, columns=['Correlation'])
    print("\nCorrelation Matrix:\n",correlation_matrix)

    plt.figure(figsize=(5, 8))  
    sb.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
    plt.title('Correlation with Target Variable')
    plt.show()

#find correlation and return top features
def corr(target_df,features_df,target_col):    
    traget_series=target_df[target_col]
    correlations=features_df.corrwith(traget_series)  

    #top 20 correlations
    top_correlations=correlations.abs().nlargest(top_col_no)  
    return top_correlations

def Plot(target_df, features_df, target_col,feature_name): 
    # Separate the data based on numeric labels
    target_label=target_df[target_col].unique()
    # print(target_label)
    for i in target_label:
        class_0 = features_df.loc[target_df[target_col] == i]
        if i==0:
            plt.plot(class_0[feature_name], np.zeros_like(class_0[feature_name]), 'x', label=f'Class {i}')
        else:
            plt.plot(class_0[feature_name], np.zeros_like(class_0[feature_name]), '|', label=f'Class {i}')

    plt.legend()
    plt.xlabel(feature_name)
    plt.title('1D Scatter Plot of '+ feature_name +' by Numeric Classes')
    plt.show()



# 1(b). Dataset1

# 1(c). Dataset2

# 1(d). Dataset3

# online

In [83]:
def online():
    dataframe=pd.read_csv("concentric_squares_dataset.csv")
    target_col='y'    #define target

    datastat(dataframe)
    
    #handle space
    # replace with null
    # print(dataframe['TotalCharges'].dtype)
    dataframe.replace(' ', np.nan, inplace=True)
    
    #missing,duplicate
    missing_duplicate(dataframe)

    #drop target cols, duplicate cols and any unnecessary cols
    dataframe=droprows(dataframe,target_col)

    dataframe=replace_str(dataframe)
    dataframe=replace_num(dataframe)
    missing=dataframe.isnull().sum()
    print("final Missing:\n",missing)

    # print(dataframe)

    #split dataframe into feature and target
    features=dataframe.drop(target_col,axis=1)
    target=dataframe[target_col]

    #conversion of feature into numeric
    # print(features.dtypes)

    #label encoding yes,no features
   
    #split
    features_numeric=features.select_dtypes(include=['number'])
    features_non_numeric=features.select_dtypes(exclude=['number'])
    non_numeric_cols = features.select_dtypes(exclude=['number']).columns
    # print(features_numeric)
    # print(features_non_numeric)
    print("non numeric cols: ",non_numeric_cols)
    
    #onehot encoding features
    # features_non_numeric=onehot(features_non_numeric, non_numeric_cols)

   
    #target into numeric
    encoder = LabelEncoder()
    target = encoder.fit_transform(target)
    # target

    #scaled feature
    scaled_features=Scaling(features=features_numeric, type=scaling_type)
    # print(scaling_type+"\n",scaled_features)    

    #corr analysis
    #merge numeric & non_numeric
    features_df=pd.DataFrame(scaled_features, columns=features_numeric.columns)
    # features_non_numeric=pd.DataFrame(features_non_numeric.to_numpy(), columns=features_non_numeric.columns)
    # features_df=pd.concat([features_df,features_non_numeric],axis=1)
    target_df=pd.DataFrame(target, columns=[target_col])

    print(features_df.shape)

    # corr
    top_correlations=corr(target_df,features_df,target_col)    
  
    # plot_corr(top_correlations)
    # print(top_correlations)
    top_correlations_features=top_correlations.index
    print(top_correlations_features.size)
    
    #scatter plot
    # for tf in top_correlations_features:
    #     Plot(target_df, features_df, target_col,tf)

    x=features_df[list(top_correlations_features)]  #f_df already scaled
    y=target_df  #cng here

    return x,y

# 1(e). Preprocess dataset

In [84]:
# x,y=pre1()
# x,y=pre2()
# x,y=pre3()
x,y=online()
# print(x.shape)
x=x.reset_index(drop=True)
y=y.reset_index(drop=True)

Number of attributes:  3
Number of records:  2000
Missing:
 X1    0
X2    0
y     0
dtype: int64
Duplicate:  0
Duplicate:  0
final Missing:
 X1    0
X2    0
y     0
dtype: int64
non numeric cols:  Index([], dtype='object')
(2000, 2)
2


# 2. LR

In [85]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
# from sklearn.linear_model import LogisticRegression
    
class LogisticRegression:
    def __init__(self,alpha=3.65, lamda=0.2) -> None:
        self.alpha=alpha
        self.lamda=lamda
        self.w=None
        self.x=None
        self.y=None
        self.hx=None
        self.sample=None
        

    def sigmoid(self,x):
        # w=np.transpose(self.w) #self.w.T
        temp=np.dot(x,self.w)#x->150,5*5,1
        # print("np.dot",x.shape,self.w.shape)
        temp=np.exp(-temp)
        temp=1+temp
        self.hx=1/temp
        # print("sig:",self.hx.shape)

        return self.hx    
    
    def loss(self):
        temp=self.y-self.sigmoid(self.x)
        # print("temp",temp.shape, self.x.shape)
        temp=np.dot(temp.T,self.x)
        # print(temp.shape)#1*n
        return temp
    
    def reg(self):
        reg=np.dot(self.w.T,self.w)-(self.w[0]*self.w[0])   #l2
        reg=(self.lamda/(2*self.sample))*reg
        #ei jinis bais bade sobar
        return reg

    def gradientDescent(self):
        err=1
        i=1

        while (err>MIN_ERR):
            err_arr=(1/self.sample)*self.alpha*self.loss()+self.reg()#1*7
            err_arr[0]=err_arr[0]-self.reg() #exclude bais
            # err_arr=(1/self.sample)*self.alpha*self.loss()#1*7
            # err_arr=self.alpha*self.loss()#1*7
            err_arr=err_arr.T#shape as self.w
            s=np.sum(self.w)
            self.w=self.w+err_arr
            # self.alpha=self.alpha*.95
            err=np.sum(abs(err_arr))
            # if(s!=0):
                # err=np.sum(abs(err_arr))/s
            
            i=i+1
            if(i>10000):
                break
            # print(err)
        # print(self.w)
        return  self.w #n*1

    def fit(self,x,y):
        self.sample=x.shape[0]
        self.x=x.to_numpy(dtype=float)
        # print(self.x.shape, x.shape)
        ones = np.ones((self.x.shape[0], 1))
        self.x=np.hstack((ones,self.x))
        # print(self.x.shape,x.shape)
        
        self.y=y.to_numpy()        
        self.y=self.y.reshape(-1,1) #y must be a col y.reshape(-1,1)
        # print(self.y.shape)
        self.w=np.zeros((self.x.shape[1],1))
        # print(self.w.shape)
        self.gradientDescent()

    def predict(self,x_test):
        testing=x_test.to_numpy(dtype=float)
        ones = np.ones((testing.shape[0], 1))
        testing=np.hstack((ones,testing))
        # print("pre",testing.shape)
        
        y_pred=self.sigmoid(testing)
        y_pred=y_pred.flatten()
        y_pred=(y_pred>=0.5).astype(int)
        return y_pred



def Prediction(x,y):
    x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=56)

    model = LogisticRegression()
    # print(x_train)
    model.fit(x_train, y_train)

    y_pred = model.predict(x_test)
    # print( y_pred)
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy


print(f"Accuracy of Logistic Regression classifier with {scaling_type}: {Prediction(x,y):.6f}")


Accuracy of Logistic Regression classifier with StandardScaler: 0.562500


# 3. Performance

In [86]:
def Eval(y_orin, y_pred):
    tn,fp,fn,tp=confusion_matrix(y_orin,y_pred).ravel()
    # print(confusion_matrix(y_orin,y_pred).ravel())

    accuracy = accuracy_score(y_orin, y_pred)
    sensitivity=tp/(tp+fn)
    specificity=tn/(tn+fp)
    precision=tp/(tp+fp)
    f1=f1_score(y_orin,y_pred)
    auroc=roc_auc_score(y_orin, y_pred)
    aupr=average_precision_score(y_orin, y_pred)

    val=[ accuracy, sensitivity, specificity, precision, f1, auroc, aupr ]
    return val



# 4. Bagging

In [87]:
from sklearn.model_selection import train_test_split
X_TRAIN, X_TEST, Y_TRAIN, Y_TEST = train_test_split(x,y, test_size=0.2, random_state=56) #x,y top corr, keep 20% for testing only
x_train, x_validation, y_train, y_validation=train_test_split(X_TRAIN,Y_TRAIN, test_size=0.2, random_state=42)

#work on x_train
x_train_bs=[]
y_train_bs=[]
base_model=[]
y_pred_bs=[]
accuracy_bs=[]

def bagging():
    for i in range(0,9):
        xtrain=x_train.sample(n=x_train.shape[0], random_state=42*(i+1), replace=True)
        ytrain=y_train.loc[xtrain.index]
        # print(ytrain)

        bm=LogisticRegression()
        bm.fit(xtrain,ytrain)
        ypred = bm.predict(x_validation)   #validation
        acc = accuracy_score(y_validation, ypred)


        x_train_bs.append(xtrain)
        y_train_bs.append(ytrain)
        base_model.append(bm)
        y_pred_bs.append(ypred)
        accuracy_bs.append(acc)
        print("Accuracy of base model on validation\n",accuracy_bs[i])
        # print(x_train_bs)


bagging()

Accuracy of base model on validation
 0.48125
Accuracy of base model on validation
 0.490625
Accuracy of base model on validation
 0.565625
Accuracy of base model on validation
 0.48125
Accuracy of base model on validation
 0.64375
Accuracy of base model on validation
 0.41875
Accuracy of base model on validation
 0.484375
Accuracy of base model on validation
 0.51875
Accuracy of base model on validation
 0.51875


# 5. Majority voting model

In [88]:
def prediction_by_base(X_TEST):  #prediction of test by base learners
    ypred_by_individual=[]
    for i in range(0,9):
        ypred = base_model[i].predict(X_TEST)   #global test data
        ypred_by_individual.append(ypred)

    return ypred_by_individual
        

def votingEnsembler(ypred_by_individual):
    y_pred_dataframe=np.array(ypred_by_individual)
    y_pred_dataframe=pd.DataFrame(y_pred_dataframe)
    # print(y_pred_dataframe)
    y_pred_voting=[]
    for i in range(y_pred_dataframe.shape[1]):
        mode=y_pred_dataframe[i].mode()[0]
        y_pred_voting.append(mode)

    y_pred_voting=np.array(y_pred_voting)
    
    majority_performance=Eval(y_orin=Y_TEST,y_pred=y_pred_voting)
    # print("Result",y_pred_voting)
    print("Majority voting performance",majority_performance)
    return majority_performance

ypred_by_individual=prediction_by_base(X_TEST)
# print(ypred_by_individual)
majority_performance=votingEnsembler(ypred_by_individual)

Majority voting performance [0.5375, np.float64(0.42718446601941745), np.float64(0.654639175257732), np.float64(0.567741935483871), np.float64(0.48753462603878117), np.float64(0.5409118206385747), np.float64(0.537530535546508)]


# 6. Stacking

In [89]:
# meta model train

# y_pred_bs pred on validation of bs
# create validation+prediction for the train data of meta
# print(y_pred_bs)
y_pred_bs_df=pd.DataFrame(y_pred_bs)
y_pred_bs_df=y_pred_bs_df.T
# print(y_pred_bs_df)
# print(x_validation.shape)
x_validation=x_validation.reset_index(drop=True)
y_pred_bs_df=pd.concat([x_validation,y_pred_bs_df],axis=1)
# print(y_pred_bs_df)

#
y_pred_bs_df.columns=y_pred_bs_df.columns.astype('str')

Meta_model=LogisticRegression()
Meta_model.fit(y_pred_bs_df,y_validation)

#meta model test

#base
base_performance=[]
ypred_for_test=[]
for i in range(0,9):
    ypred = base_model[i].predict(X_TEST)   
    ypred_for_test.append(ypred)
    base_performance.append(Eval(Y_TEST, ypred))
    # print(Eval(Y_TEST, ypred))

# print(ypred_for_test)
ypred_test_df=pd.DataFrame(ypred_for_test)
ypred_test_df=ypred_test_df.T
# print(ypred_test_df)
X_TEST=X_TEST.reset_index(drop=True)
ypred_test_df=pd.concat([X_TEST,ypred_test_df],axis=1)

#meta
ypred_test_df.columns=ypred_test_df.columns.astype('str')
final_pred=Meta_model.predict(ypred_test_df)
stacking_performance=Eval(y_orin=Y_TEST,y_pred=final_pred)
print("Meta model prediction", stacking_performance)

# print("result")
# print(ypred_test_df)
# print(final_pred) 
# print(Y_TEST) 


Meta model prediction [0.725, np.float64(0.8883495145631068), np.float64(0.5515463917525774), np.float64(0.6777777777777778), np.float64(0.7689075630252101), np.float64(0.7199479531578421), np.float64(0.6596035598705502)]


C:\Users\Binary Gadget\AppData\Local\Temp\ipykernel_9384\3734062835.py:8: RuntimeWarning: invalid value encountered in scalar divide
  precision=tp/(tp+fp)
C:\Users\Binary Gadget\AppData\Local\Temp\ipykernel_9384\3734062835.py:8: RuntimeWarning: invalid value encountered in scalar divide
  precision=tp/(tp+fp)


# 7. Performance on test

In [90]:
# print(np.array(base_performance))
#perfomance of base learners
bp_df=pd.DataFrame(base_performance, columns=['accuracy','sensitivity', 'specificity', 'precision', 'f1', 'auroc', 'aupr'])
# print(bp_df)

#voilin plot
# for i in ['accuracy','sensitivity', 'specificity', 'precision', 'f1', 'auroc', 'aupr']:
    # sb.violinplot(y=bp_df[i])
    # plt.title('Violin Plot for '+i)
    # plt.xlabel('Base Learners')
    # plt.ylabel(i)
    # plt.show()

#finding mean and std of base learners
bp_avg=[]
bp_std=[]
for i in  ['accuracy','sensitivity', 'specificity', 'precision', 'f1', 'auroc', 'aupr']:
    bp_avg.append(np.mean(bp_df[i]))
    bp_std.append(np.std(bp_df[i]))

# print(bp_avg,"\n",bp_std)

#final analysis table
def df_create(result_df,entry):
    tdf=pd.DataFrame(entry)
    tdf=tdf.T
    tdf.columns=result_df.columns
    result_df=pd.concat([result_df,tdf],axis=0)
    return result_df

result_df=pd.DataFrame(columns=['Accuracy','Sensitivity', 'Specificity', 'Precision', 'F1-score', 'AUROC', 'AUPR'])

result_df=df_create(result_df,bp_avg)
result_df=df_create(result_df,bp_std)

result_df=df_create(result_df,majority_performance)
result_df=df_create(result_df,stacking_performance)
result_df.index=['LR(Mean)','LR(Stdev)','Voting ensemble', 'Stacking ensemble']

print(result_df)


                   Accuracy  Sensitivity  Specificity  Precision  F1-score  \
LR(Mean)           0.500278     0.437433     0.567010   0.602646  0.385367   
LR(Stdev)          0.039816     0.345987     0.394426   0.178473  0.249337   
Voting ensemble    0.537500     0.427184     0.654639   0.567742  0.487535   
Stacking ensemble  0.725000     0.888350     0.551546   0.677778  0.768908   

                      AUROC      AUPR  
LR(Mean)           0.502221  0.520656  
LR(Stdev)          0.044735  0.028011  
Voting ensemble    0.540912  0.537531  
Stacking ensemble  0.719948  0.659604  


C:\Users\Binary Gadget\AppData\Local\Temp\ipykernel_9384\1798838075.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result_df=pd.concat([result_df,tdf],axis=0)
